# Excusify: fine-tuning GPT-2 for situation-aware excuses, apologies and emergency messages

This notebook trains the three models used by the Excusify app, measures how well their answers fit the
situation, and uploads the best version of each to the Hugging Face Hub.

**Run it on a GPU:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. It takes about 20–30 minutes.

How it works:

1. **Every row says what it is for.** Each example has a `Situation` (e.g. *forgot to submit the report*), so the model learns to write for that situation.
2. **The model learns when to stop.** The loss covers the completion **and** its end-of-text token. (The common setup of `pad_token = eos_token` with `DataCollatorForLanguageModeling` masks every end-of-text label, so a model trained that way never learns to stop.)
3. **Loss only on the completion.** Prompt tokens are masked with `-100`.
4. **Relevance is measured, not assumed.** Each setting in a small learning-rate/epoch sweep is scored by *situation match*: the share of generated answers that mention the situation they were asked about, on held-out prompts and on situations that are not in the data at all. The best setting per model is uploaded.
5. **Decoding doesn't fight the prompt.** `repetition_penalty` and `no_repeat_ngram_size` are off, because in transformers they also penalise words from the prompt, i.e. the situation the model is supposed to repeat.
6. **Training and the app share one prompt format.** Step 2 is copied verbatim into `app.py`.

In [ ]:
!pip install -q "transformers>=4.41" accelerate huggingface_hub scikit-learn

## 1. Configuration

In [ ]:
import os

HF_USERNAME = "Sohamb2005"     # your Hugging Face username
PUSH_TO_HUB = True             # set to False to train and test without uploading
BASE_MODEL = "gpt2"
DATA_URL = "https://raw.githubusercontent.com/soham200521/Excusify/main/data"
MAX_LENGTH = 96                # the longest example is checked against this below
BATCH_SIZE = 8
SEED = 42

# every model is trained once per setting; the setting with the best situation match is uploaded
# (on a CPU-only runtime keep just the middle setting, otherwise this takes hours)
SWEEP = [
    {"learning_rate": 5e-5, "epochs": 5},
    {"learning_rate": 1e-4, "epochs": 10},
    {"learning_rate": 1e-4, "epochs": 20},
]

MODELS = {
    "excuse": f"{HF_USERNAME}/gpt2-finetuned-excuses",
    "apology": f"{HF_USERNAME}/gpt2-finetuned-apologies",
    "emergency": f"{HF_USERNAME}/gpt2-finetuned-emergency",
}

### Hugging Face token

Paste a **write** token when the box appears (huggingface.co → Settings → Access Tokens). It is checked straight away, so a wrong token fails here and not after training.

In [ ]:
HF_TOKEN = None
if PUSH_TO_HUB:
    from getpass import getpass
    from huggingface_hub import HfApi
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Paste your Hugging Face WRITE token and press Enter: ")
    me = HfApi(token=HF_TOKEN).whoami()["name"]
    if me != HF_USERNAME:
        raise ValueError(f"This token belongs to '{me}', but HF_USERNAME is '{HF_USERNAME}'.")
    print(f"Logged in as {me}")

## 2. Prompt format, decoding and relevance (shared with `app.py`; keep the two copies identical)

In [ ]:
import re

def normalize(text):
    """Lower-case, straight quotes, no separator characters, no trailing punctuation."""
    text = (text or "").replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = re.sub(r"[|:\n]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip().rstrip(".!?").strip()
    return text.lower()


def excuse_prompt(scenario, urgency, believability, situation):
    return f"{normalize(scenario)} | {normalize(urgency)} | {normalize(believability)} | {normalize(situation)} :"


def apology_prompt(apology_type, situation):
    return f"{normalize(apology_type)} | {normalize(situation)} :"


def emergency_prompt(scenario, details=""):
    details = normalize(details)
    return f"{normalize(scenario)} | {details} :" if details else f"{normalize(scenario)} :"


# repetition_penalty and no_repeat_ngram_size stay OFF: in transformers both also count the prompt, so they
# punish the model for repeating the situation ("late" became "early", "meeting" became "gathering")
GEN_KWARGS = dict(do_sample=True, top_p=0.92, top_k=50, temperature=0.8,
                  repetition_penalty=1.0, no_repeat_ngram_size=0)

STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "was", "were", "are", "you", "your", "our", "but", "not",
    "have", "had", "has", "from", "about", "into", "all", "can", "could", "will", "would", "didn't", "don't",
    "can't", "i'm", "i've", "i'll", "she", "her", "him", "his", "they", "them", "its", "it's", "too", "very",
    "got", "get", "just", "out", "today", "tonight", "yesterday", "being", "been", "after", "before", "some",
    "then", "than", "there", "their", "when", "what", "because", "sorry", "really", "need", "make", "my",
}


def _stem(word):
    for suffix in ("ing", "ed", "es", "s"):
        if len(word) > len(suffix) + 2 and word.endswith(suffix):
            word = word[: -len(suffix)]
            break
    return word.rstrip("'")[:5]


def _stems(text):
    return {_stem(w) for w in re.findall(r"[a-z']+", (text or "").lower()) if len(w) > 2 and w not in STOPWORDS}


def relevance(candidate, target):
    """How many content words of the user's input the candidate mentions (5-letter stems)."""
    return len(_stems(target) & _stems(candidate))

## 3. Load the datasets

In [ ]:
import pandas as pd

def load_csv(name):
    local = os.path.join("data", name)
    path = local if os.path.exists(local) else f"{DATA_URL}/{name}"
    df = pd.read_csv(path, keep_default_na=False)
    print(f"{name}: {len(df)} rows  ({path})")
    return df


def to_pairs(df, prompt_fn, situation_fn, text_col, group_col):
    pairs = pd.DataFrame({
        "prompt": df.apply(prompt_fn, axis=1),
        "situation": df.apply(situation_fn, axis=1),
        "completion": df[text_col].str.strip(),
        "group": df[group_col],
    })
    before = len(pairs)
    pairs = pairs.drop_duplicates(subset=["completion"]).reset_index(drop=True)
    if before != len(pairs):
        print(f"  dropped {before - len(pairs)} duplicate rows")
    return pairs


excuses = load_csv("excuses.csv")
apologies = load_csv("apologies.csv")
emergencies = load_csv("emergency_messages.csv")

datasets = {
    "excuse": to_pairs(excuses, lambda r: excuse_prompt(r.Scenario, r.Urgency, r.Believability, r.Situation),
                       lambda r: r.Situation, "Excuse", "Scenario"),
    "apology": to_pairs(apologies, lambda r: apology_prompt(r.Type, r.Situation), lambda r: r.Situation,
                        "Apology", "Type"),
    "emergency": to_pairs(emergencies, lambda r: emergency_prompt(r.Scenario, r.Details), lambda r: r.Details,
                          "Message", "Scenario"),
}
for name, pairs in datasets.items():
    print(f"\n{name}:  {pairs.prompt[0]} {pairs.completion[0]}")

# situations that are NOT in the data, to test generalisation: (prompt, situation)
UNSEEN = {
    "excuse": [(excuse_prompt(*args), args[3]) for args in [
        ("social", "low", "high", "late for my friend's birthday dinner"),
        ("family", "medium", "high", "forgot to water the plants"),
        ("work", "medium", "high", "missed the flight for the business trip"),
        ("school", "low", "high", "forgot to charge my laptop before the online class"),
        ("work", "high", "medium", "can't join the conference call"),
        ("social", "medium", "low", "skipped the gym session with my friend"),
        ("family", "low", "low", "forgot to pick up the dry cleaning"),
        ("school", "medium", "medium", "late returning the lab equipment"),
    ]],
    "apology": [(apology_prompt(*args), args[1]) for args in [
        ("informal", "eating your leftover pizza"),
        ("professional", "missing the client presentation"),
        ("emotional", "forgetting to call you on your big day"),
        ("professional", "the delay in replying to your application"),
        ("informal", "borrowing your bike without asking"),
        ("emotional", "missing your farewell party"),
    ]],
    "emergency": [(emergency_prompt(*args), args[1]) for args in [
        ("stuck somewhere", "stuck at the airport after a cancelled flight"),
        ("car trouble", "car battery failed in the office parking"),
        ("medical issue", "sprained wrist at the gym"),
        ("work issue", "the client portal is down"),
        ("school absence", "train strike in the city"),
        ("urgent help needed", "locked out of my hostel room"),
    ]],
}

## 4. Tokenise with prompt masking and an end-of-text label

In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token


class PromptCompletionDataset(Dataset):
    """Loss is computed on the completion and its EOS token only; prompt and padding are -100."""

    def __init__(self, frame):
        self.items = [self.encode(p, c) for p, c in zip(frame.prompt, frame.completion)]

    @staticmethod
    def encode(prompt, completion):
        prompt_ids = tokenizer(prompt)["input_ids"]
        completion_ids = tokenizer(" " + completion)["input_ids"] + [tokenizer.eos_token_id]
        ids = prompt_ids + completion_ids
        if len(ids) > MAX_LENGTH:
            raise ValueError(f"Example is {len(ids)} tokens, above MAX_LENGTH={MAX_LENGTH}: {prompt} {completion}")
        pad = MAX_LENGTH - len(ids)
        return {
            "input_ids": torch.tensor(ids + [tokenizer.pad_token_id] * pad),
            "attention_mask": torch.tensor([1] * len(ids) + [0] * pad),
            "labels": torch.tensor([-100] * len(prompt_ids) + completion_ids + [-100] * pad),
        }

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


splits = {}
for name, pairs in datasets.items():
    train_df, eval_df = train_test_split(pairs, test_size=0.1, random_state=SEED, stratify=pairs["group"])
    train_ds, eval_ds = PromptCompletionDataset(train_df), PromptCompletionDataset(eval_df)
    longest = max(int(x["attention_mask"].sum()) for x in train_ds.items + eval_ds.items)
    splits[name] = (train_ds, eval_ds, eval_df)
    print(f"{name:9s}  train={len(train_ds):4d}  eval={len(eval_ds):3d}  longest example={longest} tokens")

first = splits["excuse"][0][0]
assert tokenizer.eos_token_id in first["labels"].tolist(), "EOS must be a training target"
print("EOS is a training target:", True)

## 5. Train every setting and keep the one whose answers fit the situation best

*Situation match* = share of generated answers that mention at least one content word of the situation they were asked for.
`match_eval` uses held-out prompts, `match_unseen` uses situations that are not in the data, and `gold_match` is the
same measure on the hand-written held-out answers (a rough ceiling).

Selection rule: among the settings whose combined match is within 0.10 of the best, keep the one with the lowest
held-out loss (the least over-fitted model that is still about as relevant).

In [ ]:
import gc
import inspect
import math
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, default_data_collator, set_seed


def make_args(output_dir, learning_rate, epochs):
    kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available(),
    )
    # argument names differ between transformers 4.x and 5.x
    params = inspect.signature(TrainingArguments.__init__).parameters
    kwargs["eval_strategy" if "eval_strategy" in params else "evaluation_strategy"] = "no"
    if "warmup_ratio" in params:
        kwargs["warmup_ratio"] = 0.1      # transformers 4.x
    else:
        kwargs["warmup_steps"] = 0.1      # transformers 5.x: a float below 1 is a ratio
    return TrainingArguments(**kwargs)


def generate(model, prompt, n=3, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, num_return_sequences=n, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id, **GEN_KWARGS)
    start = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[start:], skip_special_tokens=True).strip() for o in out]


def situation_match(model, items, n=2):
    """items: (prompt, situation) pairs. Share of samples that mention the situation."""
    hits = total = 0
    for prompt, situation in items:
        if not _stems(situation):
            continue
        for text in generate(model, prompt, n=n):
            total += 1
            hits += relevance(text, situation) > 0
    return round(hits / max(total, 1), 3)


def make_trainer(name, model, learning_rate=5e-5, epochs=1):
    train_ds, eval_ds, _ = splits[name]
    return Trainer(model=model, args=make_args(f"./runs/{name}", learning_rate, epochs), train_dataset=train_ds,
                   eval_dataset=eval_ds, data_collator=default_data_collator)


def load_base():
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
    model.config.pad_token_id = tokenizer.eos_token_id
    return model


results, sweep_log = {}, []
for name in ["excuse", "apology", "emergency"]:
    train_ds, eval_ds, eval_df = splits[name]
    eval_items = list(zip(eval_df.prompt, eval_df.situation))
    gold = [relevance(c, s) > 0 for c, s in zip(eval_df.completion, eval_df.situation) if _stems(s)]
    base_loss = make_trainer(name, load_base()).evaluate()["eval_loss"]
    runs = []
    for setting in SWEEP:
        set_seed(SEED)
        trainer = make_trainer(name, load_base(), setting["learning_rate"], setting["epochs"])
        trainer.train()
        loss = trainer.evaluate()["eval_loss"]
        trainer.optimizer = trainer.lr_scheduler = None   # free memory; only the model is needed now
        model = trainer.model.eval()
        row = {"model": name, **setting, "eval_loss": round(loss, 3), "perplexity": round(math.exp(loss), 1),
               "match_eval": situation_match(model, eval_items), "match_unseen": situation_match(model, UNSEEN[name])}
        sweep_log.append(row)
        runs.append((trainer, row))
        print(row)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    best_match = max(r["match_eval"] + r["match_unseen"] for _, r in runs)
    trainer, row = min((run for run in runs if run[1]["match_eval"] + run[1]["match_unseen"] >= best_match - 0.10),
                       key=lambda run: run[1]["eval_loss"])
    del runs
    gc.collect()
    results[name] = (trainer, {
        **row, "train_examples": len(train_ds), "eval_examples": len(eval_ds),
        "base_eval_loss": round(base_loss, 3), "base_perplexity": round(math.exp(base_loss), 1),
        "gold_match": round(sum(gold) / max(len(gold), 1), 3),
    })
    print(f"==> {name}: keeping learning_rate={row['learning_rate']}, epochs={row['epochs']}\n")

## 6. Try the chosen models

`raw` shows three samples; `app` shows what the app would display: it samples 6, keeps those that mention the situation most, and of those picks the one the model finds most likely for this prompt.

In [ ]:
def completion_logprobs(model, tokenizer, prompt, candidates):
    """Mean log-probability per token of each candidate (plus end-of-text) given the prompt."""
    prompt_ids = tokenizer(prompt)["input_ids"]
    scores = []
    with torch.no_grad():
        for text in candidates:
            ids = prompt_ids + tokenizer(" " + text)["input_ids"] + [tokenizer.eos_token_id]
            logits = model(torch.tensor([ids], device=model.device)).logits[0, :-1].float()
            positions = torch.arange(len(ids) - 1, device=logits.device)
            targets = torch.tensor(ids[1:], device=logits.device)
            token_logp = torch.log_softmax(logits, dim=-1)[positions, targets]
            scores.append(token_logp[len(prompt_ids) - 1:].mean().item())
    return scores


def app_pick(model, prompt, situation, n=6):
    candidates = list(dict.fromkeys(c for c in generate(model, prompt, n=n) if len(c.split()) >= 4))
    if not candidates:
        return ""
    rel = [relevance(c, situation) for c in candidates]
    top = [c for c, r in zip(candidates, rel) if r == max(rel)]
    scores = completion_logprobs(model, tokenizer, prompt, top)
    return top[max(range(len(top)), key=scores.__getitem__)]


for name in results:
    model = results[name][0].model.eval()
    print(f"\n===== {name} =====")
    eval_df = splits[name][2]
    examples = list(zip(eval_df.prompt, eval_df.situation))[:3] + UNSEEN[name][:4]
    for prompt, situation in examples:
        print(prompt)
        for text in generate(model, prompt):
            print("   raw ->", text)
        print("   app ->", app_pick(model, prompt, situation))

## 7. Save, write model cards and upload to the Hub

In [ ]:
from transformers import GenerationConfig

DESCRIPTIONS = {
    "excuse": ("excuses", "an excuse for a situation, conditioned on scenario, urgency and believability",
               "work | medium | high | forgot to submit the report :", "excuses.csv",
               "Scenario (work/school/family/social), Situation, Urgency (low/medium/high), Believability (low/medium/high), Excuse"),
    "apology": ("apologies", "an apology for a situation in an emotional, professional or informal tone",
                "professional | being late to the meeting :", "apologies.csv",
                "Type (emotional/professional/informal), Situation, Apology"),
    "emergency": ("emergency messages", "a short, first-person urgent message for a scenario and optional details",
                  "car trouble | flat tyre on the highway :", "emergency_messages.csv",
                  "Scenario (8 types), Details (optional), Message"),
}


def model_card(name, m):
    what, task, prompt, csv_name, columns = DESCRIPTIONS[name]
    return f"""---
license: mit
language: en
base_model: gpt2
pipeline_tag: text-generation
tags: [gpt2, text-generation, fine-tuned, excusify]
---

# GPT-2 fine-tuned for {what} (Excusify)

Part of [Excusify](https://github.com/soham200521/Excusify). Given a short prompt, the model writes {task}.

## Prompt format

```
{prompt}
```

The model writes the completion after the colon and stops with an end-of-text token.

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

repo = "{MODELS[name]}"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo)

inputs = tokenizer("{prompt}", return_tensors="pt")
out = model.generate(**inputs, do_sample=True, top_p=0.92, temperature=0.8, max_new_tokens=60,
                     pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

## Training

| | |
|---|---|
| Base model | `gpt2` (124M parameters) |
| Data | [`data/{csv_name}`](https://github.com/soham200521/Excusify/tree/main/data): {columns} |
| Split | {m["train_examples"]} train / {m["eval_examples"]} held-out examples |
| Objective | causal LM loss on the completion and its end-of-text token (prompt masked) |
| Settings | learning rate {m["learning_rate"]}, {m["epochs"]} epochs, batch {BATCH_SIZE}, cosine schedule; chosen from a sweep (best situation match, then lowest loss) |

## Evaluation

| | Value |
|---|---|
| Held-out loss (GPT-2 before → this model) | {m["base_eval_loss"]} → {m["eval_loss"]} |
| Held-out perplexity (GPT-2 before → this model) | {m["base_perplexity"]} → {m["perplexity"]} |
| Situation match, held-out prompts | {m["match_eval"]:.0%} (hand-written answers: {m["gold_match"]:.0%}) |
| Situation match, situations not in the data | {m["match_unseen"]:.0%} |

*Situation match* is the share of sampled answers that mention at least one content word of the situation in the prompt.

## Limitations

Made for entertainment and practice messages. The training set is small and hand-written, so the model can repeat
phrasings from it, and situations far from the training data may get generic answers.
"""


if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)

for name, (trainer, metrics) in results.items():
    out_dir = "./" + MODELS[name].split("/")[-1]
    trainer.model.generation_config = GenerationConfig(**GEN_KWARGS, max_new_tokens=60,
                                                       pad_token_id=tokenizer.eos_token_id,
                                                       eos_token_id=tokenizer.eos_token_id)
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "README.md"), "w") as f:
        f.write(model_card(name, metrics))
    print("saved", out_dir)
    if PUSH_TO_HUB:
        api.create_repo(MODELS[name], exist_ok=True)
        api.upload_folder(folder_path=out_dir, repo_id=MODELS[name], ignore_patterns=["training_args.bin"],
                          commit_message="Situation-aware data, EOS fix, relevance-based model selection")
        print("uploaded", MODELS[name])

## 8. Tables for the README

In [ ]:
print("Results")
print()
print("| Model | Train / eval | Perplexity (GPT-2 → fine-tuned) | Situation match: held-out / unseen (hand-written) |")
print("|---|---|---|---|")
for name, (_, m) in results.items():
    print(f"| `{MODELS[name]}` | {m['train_examples']} / {m['eval_examples']} | {m['base_perplexity']} → {m['perplexity']} | "
          f"{m['match_eval']:.0%} / {m['match_unseen']:.0%} ({m['gold_match']:.0%}) |")

print()
print("Sweep")
print()
print("| Model | Learning rate | Epochs | Eval loss | Situation match held-out | Situation match unseen |")
print("|---|---|---|---|---|---|")
for r in sweep_log:
    print(f"| {r['model']} | {r['learning_rate']} | {r['epochs']} | {r['eval_loss']} | {r['match_eval']:.0%} | {r['match_unseen']:.0%} |")